# ToS;DR Sentence and Multi-label Classification Using LEGAL-BERT

This notebook runs **offline inference** with the Lawgic fine-tuned LEGAL-BERT classifier. The first experiment is a zero-shot cross-dataset sanity check: clauses annotated as mandatory arbitration (`arb`) in the academic **100_ToS** dataset should overlap with the ToS;DR topic **Dispute Resolution**.

## 1. Imports, Paths, and Constants

Centralize configuration so the notebook can be run from the repo root or from `notebooks/tos_dr/`. The project root is discovered by walking upward until `generated_files/tos_dr/tos_dr_points.csv` is found.

In [5]:
import re
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Inference hyperparameters (aligned with legal_bert_finetuning.ipynb)
MAX_LENGTH = 256
CONFIDENCE_THRESHOLD = 0.5
NUM_TEST_CLAUSES = 34
TARGET_100_TOS_CODE = "arb"  # Mandatory arbitration in the 100_ToS taxonomy
EXPECTED_TOSDR_TOPIC = "Dispute Resolution"


def find_project_root(start: Path | None = None) -> Path:
    """Walk up from the current working directory until the Lawgic repo root is found."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "generated_files/tos_dr/tos_dr_points.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find project root containing generated_files/tos_dr/tos_dr_points.csv"
    )


PROJECT_ROOT = find_project_root()
MODEL_CHECKPOINT_DIR = PROJECT_ROOT / "saved_models/lawgic_classifier_legal-bert"
# 100_ToS clause annotations (user-facing path: generated_files/cleaned_tos_comments.csv)
COMMENTS_CSV_PATH = PROJECT_ROOT / "generated_files/100_tos/cleaned_tos_comments.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Model checkpoint: {MODEL_CHECKPOINT_DIR}")
print(f"100_ToS comments CSV: {COMMENTS_CSV_PATH}")
print(f"Inference device: {DEVICE}")

Project root: C:\Users\Enrique\Coding Projects\Thesis\lawgic
Model checkpoint: C:\Users\Enrique\Coding Projects\Thesis\lawgic\saved_models\lawgic_classifier_legal-bert
100_ToS comments CSV: C:\Users\Enrique\Coding Projects\Thesis\lawgic\generated_files\100_tos\cleaned_tos_comments.csv
Inference device: cuda


## 2. Load Fine-Tuned Model and Tokenizer

Load the saved sequence-classification head and tokenizer from `saved_models/lawgic_classifier_legal-bert/`. The checkpoint must contain Hugging Face `config.json` with `problem_type="multi_label_classification"` and the 25 ToS;DR label mappings.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT_DIR, use_fast=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT_DIR)
model.to(DEVICE)
model.eval()  # Disable dropout / batch-norm updates for deterministic inference

# Build an index -> label lookup from the saved model config
id2label = {int(label_id): label_name for label_id, label_name in model.config.id2label.items()}
num_labels = model.config.num_labels

print(f"Loaded model with {num_labels} labels from: {MODEL_CHECKPOINT_DIR}")
print(f"Problem type: {model.config.problem_type}")
print(f"Expected overlap topic present: {EXPECTED_TOSDR_TOPIC in id2label.values()}")

Loaded model with 25 labels from: C:\Users\Enrique\Coding Projects\Thesis\lawgic\saved_models\lawgic_classifier_legal-bert
Problem type: multi_label_classification
Expected overlap topic present: True


## 3. Load and Filter 100_ToS `arb` Clauses

The `comment` column in `cleaned_tos_comments.csv` encodes the annotator's variable code (e.g. `arb -1`, `serv_chg 0`). We isolate rows whose code is exactly `arb` (mandatory arbitration) using a word-boundary regex so tokens like `arbitration` are not matched.

We then take the first 20 matching clauses as a fixed sanity-check sample.

In [7]:
comments_df = pd.read_csv(COMMENTS_CSV_PATH)

# Match the standalone variable code "arb" inside annotation strings such as "arb -1"
arb_code_pattern = re.compile(rf"\b{re.escape(TARGET_100_TOS_CODE)}\b", flags=re.IGNORECASE)

arb_mask = comments_df["comment"].astype(str).apply(lambda text: bool(arb_code_pattern.search(text)))
arb_df = comments_df.loc[arb_mask].copy()

if len(arb_df) < NUM_TEST_CLAUSES:
    raise ValueError(
        f"Expected at least {NUM_TEST_CLAUSES} `{TARGET_100_TOS_CODE}` clauses, found {len(arb_df)}."
    )

# Fixed sample for reproducible cross-dataset evaluation
test_sample_df = arb_df.head(NUM_TEST_CLAUSES).reset_index(drop=True)

print(f"Total `{TARGET_100_TOS_CODE}` clauses in dataset: {len(arb_df)}")
print(f"Sanity-check sample size: {len(test_sample_df)}")
display(test_sample_df[["company", "comment", "referenced_text"]].head(3))

Total `arb` clauses in dataset: 34
Sanity-check sample size: 34


,company,comment,referenced_text
0,SQUARE ENIX,arb -1,Binding Arbitration and Waiver of Class Actions
1,Spotify,arb -1,any claim arising under these terms must be co...
2,Spotify,arb -1,mandatory arbitration


## 4. Zero-Shot Cross-Dataset Inference

For each clause:

1. Tokenize `referenced_text` (the actual ToS clause span).
2. Run a forward pass under `torch.no_grad()`.
3. Apply **sigmoid** independently to each logit (correct for multi-label classification; softmax would incorrectly force probabilities to sum to 1).
4. Flag any label whose sigmoid probability exceeds `0.5`.

In [8]:
def predict_flagged_labels(clause_text: str, threshold: float = CONFIDENCE_THRESHOLD) -> list[tuple[str, float]]:
    """Return (label, confidence) pairs that exceed the multi-label decision threshold."""
    encoded = tokenizer(
        clause_text,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}

    with torch.no_grad():
        logits = model(**encoded).logits.squeeze(0)

    # Sigmoid: each of the 25 labels is scored independently in [0.0, 1.0]
    probabilities = torch.sigmoid(logits).cpu().numpy()

    flagged = [
        (id2label[label_idx], float(probabilities[label_idx]))
        for label_idx in range(num_labels)
        if probabilities[label_idx] >= threshold
    ]
    flagged.sort(key=lambda item: item[1], reverse=True)
    return flagged


def truncate_for_display(text: str, max_chars: int = 140) -> str:
    cleaned = " ".join(str(text).split())
    if len(cleaned) <= max_chars:
        return cleaned
    return cleaned[: max_chars - 3].rstrip() + "..."


print(f"--- LAWGIC CROSS-DATASET SANITY CHECK: `{TARGET_100_TOS_CODE}` to `{EXPECTED_TOSDR_TOPIC}` ---\n")

dispute_resolution_hits = 0

for clause_idx, row in test_sample_df.iterrows():
    clause_text = str(row["referenced_text"])
    flagged_labels = predict_flagged_labels(clause_text)

    if any(label == EXPECTED_TOSDR_TOPIC for label, _ in flagged_labels):
        dispute_resolution_hits += 1

    print(f'Test Clause {clause_idx + 1}: "{truncate_for_display(clause_text)}"')
    print("👉 FLAGGED LABELS:")

    if flagged_labels:
        for label_name, confidence in flagged_labels:
            print(f"  - {label_name} (Confidence: {confidence:.4f})")
    else:
        print("  - [CLEAR] Model failed to trigger any flags > 0.5 threshold.")

    print()

print(
    f"Summary: {dispute_resolution_hits}/{len(test_sample_df)} clauses flagged `{EXPECTED_TOSDR_TOPIC}`."
)

--- LAWGIC CROSS-DATASET SANITY CHECK: `arb` to `Dispute Resolution` ---

Test Clause 1: "Binding Arbitration and Waiver of Class Actions"
👉 FLAGGED LABELS:
  - Dispute Resolution (Confidence: 0.9794)

Test Clause 2: "any claim arising under these terms must be commenced (by filing a demand for arbitration or filing an individual action under the arbitr..."
👉 FLAGGED LABELS:
  - Dispute Resolution (Confidence: 0.9791)

Test Clause 3: "mandatory arbitration"
👉 FLAGGED LABELS:
  - Dispute Resolution (Confidence: 0.9778)

Test Clause 4: "If you are using the Services on behalf of a business (rather than for your personal use), you and Snap Group Limited agree that to the e..."
👉 FLAGGED LABELS:
  - Dispute Resolution (Confidence: 0.9820)

Test Clause 5: "The exclusive means of resolving any dispute or claim arising out of or relating to this Agreement (including any alleged breach thereof)..."
👉 FLAGGED LABELS:
  - Dispute Resolution (Confidence: 0.9797)

Test Clause 6: "If you ever wish 